In [1]:
from bs4 import BeautifulSoup
import requests
import csv
import re
import math
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
import nltk
from nltk.corpus import stopwords


In [ ]:
# Define the headers for the HTTP request
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0'
}

# Request the IMDb Top 250 page
url = requests.get("https://www.imdb.com/chart/top/", headers=headers)
url.raise_for_status()
soup = BeautifulSoup(url.content, "html.parser")

# Find the list of movies
movies_list = soup.find('ul', {'class':'ipc-metadata-list ipc-metadata-list--dividers-between sc-a1e81754-0 eBRbsI compact-list-view ipc-metadata-list--base'}).find_all('li',{'class':'ipc-metadata-list-summary-item sc-10233bc-0 iherUv cli-parent'})

# Open a CSV file to write the movie details
with open('imdb_inf.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Rank', 'Movie Name', 'Year', 'Rating', 'Summary'])

    for movie in movies_list:
        rank = movie.find('h3',class_='ipc-title__text').get_text(strip=True).split(".")[0]
        movie_name = " ".join(movie.find('div',class_='ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-b189961a-9 iALATN cli-title').a.text.split()[1:])
        year = movie.find('span',class_='sc-b189961a-8 kLaxqf cli-title-metadata-item').text.strip("()")
        rating = movie.find('span',class_='ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating').text.split()[0]
        summary_link = movie.find('a',class_='ipc-title-link-wrapper').get("href")
        
        # Request the movie's summary page
        summary_link_page = requests.get(f"https://www.imdb.com{summary_link}", headers=headers)
        summary_soup = BeautifulSoup(summary_link_page.content, "html.parser")
        summary = summary_soup.find('span',class_='sc-7193fc79-2 kpMXpM').text


        # Write the movie details to the CSV file
        writer.writerow([rank, movie_name, year, rating, summary])


In [2]:
# Define a list of stop words
stop_words = set([
    "a", "an", "and", "the", "is", "in", "it", "of", "to", "with", "that", "this",
    "for", "as", "on", "at", "by", "but", "or", "from", "be", "which", "if", "not",
    "are", "was", "were", "can", "has", "had", "will", "would", "there", "their", "what"
])

# Ensure you have downloaded the NLTK stopwords
nltk.download('stopwords')

# Define stop words using NLTK's stopwords
stop_words_nltk = stopwords.words('english')


def cleaner(text):
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert text to lowercase
    text = text.lower()
    # Remove stop words
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Read the CSV file and clean the summaries
with open('imdb_inf.csv', 'r', newline='', encoding='utf-8',errors='replace') as file:
    reader = csv.reader(file)
    rows = list(reader)
    header = rows[0]
    data = rows[1:]

    # Clean the summaries
    cleaned_data = []
    cleaned_summaries = []
    for row in data:
        row[4] = cleaner(row[4])
        cleaned_data.append(row)
        cleaned_summaries.append(row[4])

# Write the cleaned data back to a new CSV file
with open('clean.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(cleaned_data)

# Print the contents of the clean.csv file
with open('clean.csv', 'r', newline='', encoding='utf-8') as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

['movie_rank', 'movie_name', 'movie_year', 'movie_rating', 'summary']
['1', 'The Shawshank Redemption', '1994', '9.3', 'over course several years two convicts form friendship seeking consolation eventually redemption through basic compassion']
['2', 'The Godfather', '1972', '9.2', 'aging patriarch organized crime dynasty transfers control his clandestine empire his reluctant son']
['3', 'The Dark Knight', '2008', '9', 'when menace known joker wreaks havoc chaos people gotham batman must accept one greatest psychological physical tests his ability fight injustice']
['4', 'The Godfather Part II', '1974', '9', 'early life career vito corleone s new york city portrayed while his son michael expands tightens his grip family crime syndicate']
['5', '12 Angry Men', '1957', '9', 'jury new york city murder trial frustrated single member whose skeptical caution forces them more carefully consider evidence before jumping hasty verdict']
['6', "Schindler's List", '1993', '9', 'germanoccupied polan

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ali\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:

# Calculate term frequency (TF)
def com_tf(text):
    tf_dict = defaultdict(int)
    words = text.split()
    for word in words:
        tf_dict[word] += 1
    tf_dict = {word: count / len(words) for word, count in tf_dict.items()}
    return tf_dict

# Calculate inverse document frequency (IDF)
def comp_idf(documents):
    N = len(documents)
    idf_dict = defaultdict(int)
    for document in documents:
        for word in set(document.split()):
            idf_dict[word] += 1
    idf_dict = {word: math.log(N / count)+1 for word, count in idf_dict.items()}
    return idf_dict

# Compute TF-IDF
def comp_tfidf(documents):
    tfidf_documents = []
    idf_dict = comp_idf(documents)
    for document in documents:
        tf_dict = com_tf(document)
        tfidf_dict = {word: tf * idf_dict[word] for word, tf in tf_dict.items()}
        tfidf_documents.append(tfidf_dict)
    return tfidf_documents

# Get the cleaned summaries as TF-IDF vectors
tfidf_vectors = comp_tfidf(cleaned_summaries)

# Print the TF-IDF vectors for inspection
for i, tfidf in enumerate(tfidf_vectors):
    print(f"Movie {i+1} TF-IDF: {tfidf}")

Movie 1 TF-IDF: {'over': 0.3153134299089461, 'course': 0.38855424915348674, 'several': 0.3423444371161571, 'years': 0.2882824227017351, 'two': 0.21027767252505147, 'convicts': 0.4347640611908164, 'form': 0.3615232419462758, 'friendship': 0.38855424915348674, 'seeking': 0.38855424915348674, 'consolation': 0.4347640611908164, 'eventually': 0.4347640611908164, 'redemption': 0.38855424915348674, 'through': 0.30503671792046216, 'basic': 0.4347640611908164, 'compassion': 0.38855424915348674}
Movie 2 TF-IDF: {'aging': 0.3950128120571043, 'patriarch': 0.5016508398355574, 'organized': 0.5016508398355574, 'crime': 0.34169379816787776, 'dynasty': 0.5016508398355574, 'transfers': 0.5016508398355574, 'control': 0.4483318259463309, 'his': 0.2844049359077447, 'clandestine': 0.5016508398355574, 'empire': 0.3778479234944728, 'reluctant': 0.5016508398355574, 'son': 0.3105041744672497}
Movie 3 TF-IDF: {'when': 0.14551071500297713, 'menace': 0.31054575799344025, 'known': 0.2582308871044827, 'joker': 0.310

In [4]:
def cos_sim(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0 or np.isnan(dot_product) or np.isinf(dot_product) or np.isnan(norm_vec1) or np.isinf(norm_vec1) or np.isnan(norm_vec2) or np.isinf(norm_vec2):
        return 0  
    else:
        return dot_product / (norm_vec1 * norm_vec2)

# Convert TF-IDF dictionaries to vectors
def dict_to_vector(tfidf_dict, vocab):
    return np.array([tfidf_dict.get(word, 0) for word in vocab])

# Build vocabulary and TF-IDF matrix
def voc_tfidfm(cleaned_data):
    summaries = [row[4] for row in cleaned_data]
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(summaries)
    vocab = vectorizer.get_feature_names_out()
    tfidf_vectors = X.toarray()
    return vocab, tfidf_vectors

# Implement KNN algorithm
def knn(input_vector, tfidf_matrix, k=5):
    similarities = [cos_sim(input_vector, vector) for vector in tfidf_matrix]
    sorted_indices = np.argsort(similarities)[::-1]
    return sorted_indices[:k]

def get_similar_movies(input_summary, tfidf_matrix, vocabulary, cleaned_data, k=5):
    cleaned_input_summary = cleaner(input_summary)
    input_tfidf = TfidfVectorizer(vocabulary=vocabulary).fit_transform([cleaned_input_summary]).toarray()[0]
    input_vector = dict_to_vector({vocabulary[i]: input_tfidf[i] for i in range(len(vocabulary))}, vocabulary)
    neighbors = knn(input_vector, tfidf_matrix, k)
    return [cleaned_data[idx] for idx in neighbors]

vocabulary, tfidf_matrix = voc_tfidfm(cleaned_data)

new_plot_summary = "when earth becomes uninhabitable future farmer exnasa pilot joseph cooper tasked pilot spacecraft along team researchers find new planet humans"
similar_movies = get_similar_movies(new_plot_summary, tfidf_matrix, vocabulary, cleaned_data)

print("Movies similar to the new plot summary:")
for movie in similar_movies:
    print(movie)

Movies similar to the new plot summary:
['20', 'Interstellar', '2014', '8.7', 'when earth becomes uninhabitable future farmer exnasa pilot joseph cooper tasked pilot spacecraft along team researchers find new planet humans']
['96', '2001: A Space Odyssey', '1968', '8.3', 'after uncovering mysterious artifact buried beneath lunar surface spacecraft sent jupiter find its origins spacecraft manned two men supercomputer hal']
['175', 'Catch Me If You Can', '2002', '8.1', 'barely yet frank skilled forger who passed doctor lawyer pilot fbi agent carl becomes obsessed tracking down con man who only revels pursuit']
['181', 'Bacheha-Ye aseman', '1997', '8.2', 'after boy loses his sisters pair shoes he goes series adventures order find them when he cant he tries new way win new pair']
['183', 'Blade Runner', '1982', '8.1', 'blade runner must pursue terminate four replicants who stole ship space have returned earth find creator']


In [5]:
# Example usage
new_plot_summary = "insomniac office worker devilmaycare soap maker form underground fight club evolves into much more"

# Step 1: Clean the input summary
cleaned_input_summary = cleaner(new_plot_summary)
print(f"Cleaned input summary: {cleaned_input_summary}")

# Step 2: Compute TF-IDF for the cleaned input summary
input_tfidf = comp_tfidf([cleaned_input_summary])[0]
print(f"Input TF-IDF: {input_tfidf}")


# Step 3: Convert TF-IDF dictionary to vector
input_vector = dict_to_vector(input_tfidf, vocabulary)
print(f"Input vector: {input_vector}")
# Step 4: Find k nearest neighbors
k = 5
neighbors = knn(input_vector, tfidf_matrix, k)
print(f"The {k} nearest neighbors are indices: {neighbors}")

Cleaned input summary: insomniac office worker devilmaycare soap maker form underground fight club evolves into much more
Input TF-IDF: {'insomniac': 0.07142857142857142, 'office': 0.07142857142857142, 'worker': 0.07142857142857142, 'devilmaycare': 0.07142857142857142, 'soap': 0.07142857142857142, 'maker': 0.07142857142857142, 'form': 0.07142857142857142, 'underground': 0.07142857142857142, 'fight': 0.07142857142857142, 'club': 0.07142857142857142, 'evolves': 0.07142857142857142, 'into': 0.07142857142857142, 'much': 0.07142857142857142, 'more': 0.07142857142857142}
Input vector: [0. 0. 0. ... 0. 0. 0.]
The 5 nearest neighbors are indices: [ 12  31 236 207   0]


In [6]:
# Extract and clean summaries
summaries = [row[4] for row in data]

# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words=stop_words_nltk)
X = vectorizer.fit_transform(summaries)

# Train KNN model using scikit-learn
knn_model = NearestNeighbors(n_neighbors=5, metric='cosine').fit(X)

# Function to find similar movies using scikit-learn
def find_similar_movies_scikit(input_summary, vectorizer, knn_model, k=5):
    input_vector = vectorizer.transform([input_summary])
    distances, indices = knn_model.kneighbors(input_vector, n_neighbors=k)
    return indices[0]

# Measure time for scikit-learn implementation
new_plot_summary = "when earth becomes uninhabitable future farmer exnasa pilot joseph cooper tasked pilot spacecraft along team researchers find new planet humans"
start_time = time.time()
similar_movies_indices_scikit = find_similar_movies_scikit(new_plot_summary, vectorizer, knn_model, k=5)
scikit_time = time.time() - start_time

print(f"scikit-learn implementation time: {scikit_time:.4f} seconds")
print("Movies similar to the new plot summary using scikit-learn:")
for idx in similar_movies_indices_scikit:
    print(data[idx])

scikit-learn implementation time: 0.0020 seconds
Movies similar to the new plot summary using scikit-learn:
['20', 'Interstellar', '2014', '8.7', 'when earth becomes uninhabitable future farmer exnasa pilot joseph cooper tasked pilot spacecraft along team researchers find new planet humans']
['175', 'Catch Me If You Can', '2002', '8.1', 'barely yet frank skilled forger who passed doctor lawyer pilot fbi agent carl becomes obsessed tracking down con man who only revels pursuit']
['96', '2001: A Space Odyssey', '1968', '8.3', 'after uncovering mysterious artifact buried beneath lunar surface spacecraft sent jupiter find its origins spacecraft manned two men supercomputer hal']
['183', 'Blade Runner', '1982', '8.1', 'blade runner must pursue terminate four replicants who stole ship space have returned earth find creator']
['30', 'Star Wars', '1977', '8.6', 'luke skywalker joins forces jedi knight cocky pilot wookiee two droids save galaxy empires worlddestroying battle station while also 